# Memory Native — speed recovery: root cause of the char-LM 4x slowdown (A100)

Reproduces `research/results/speed_recovery_2026_10_18/`:

1. kernel-level profile of the char-LM step for dense AdamW and the counter layer kinds
   (top CUDA ops, top CPU ops, CPU-vs-GPU attribution, launch counts);
2. A/B of the layer implementations (torch reference / packed / decode-in-GEMM /
   decode-to-dense+cuBLAS) including a bitwise check of the new decode kernel;
3. bit-level update verification (decimation contract, one-update-per-step, exact
   no-op at lr=0);
4. the real char-LM gate: dense AdamW vs counter_triton vs counter_fast vs counter_rms,
   two seeds, peak/persistent memory, and the peak-memory gate.

Reference run: A100-SXM4-40GB, torch 2.11.0+cu130, CUDA 13.0, Triton 3.6.0. Read
`REPORT.md` for the analysis. Needs a GPU runtime.

In [ ]:
import subprocess, sys, torch
print(subprocess.run(["bash","-lc","nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"],
                     capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| triton", end=" ")
import triton; print(triton.__version__)

In [ ]:
COMMIT = "b5657f6"
print(subprocess.run(["bash","-lc",f"""
set -e
cd /content
[ -d memory-native ] || git clone -q https://github.com/kharkilirov1/memory-native.git
cd memory-native
git fetch -q origin research/speed-recovery || true
git checkout -q {COMMIT}
git log --oneline -1
python -m pip install -q -e . 2>&1 | tail -1
"""], capture_output=True, text=True).stdout)

In [ ]:
# 1) kernel-level profile: dense vs the counter kinds (~3-4 min)
print(subprocess.run(["bash","-lc","cd /content/memory-native && python scripts/profile_charlm_step.py "
  "--kinds dense,counter_rms,counter_triton,counter_fast --steps 15 --batch 32 "
  "--out-dir /tmp/sr_prof 2>&1 | grep -E '===|counter layers|attribution'"],
  capture_output=True, text=True).stdout)

In [ ]:
# 2) single-layer A/B + decode-kernel correctness (bitwise)
print(subprocess.run(["bash","-lc","cd /content/memory-native && python scripts/probe_fast_paths.py "
  "--out-dir /tmp/sr_probe 2>&1 | tail -10"], capture_output=True, text=True).stdout)

In [ ]:
# 3) bit-level update verification (decimation contract, one update/step, exact lr=0 no-op)
print(subprocess.run(["bash","-lc","cd /content/memory-native && python scripts/verify_counter_semantics.py "
  "2>&1 | grep -E 'checks:|zero_lr|decimation_restricted|one_update' | head -8"],
  capture_output=True, text=True).stdout)

In [ ]:
# 4) real training gate: same arch/data/budget, 2 seeds (~3 min)
for seed in (0, 1):
    print(subprocess.run(["bash","-lc",f"""
cd /content/memory-native && memory-native-charlm --config tiny --steps 600 --batch 32 \
  --eval-iters 20 --kinds dense,counter_triton,counter_fast,counter_rms --device cuda \
  --seed {seed} 2>&1 | grep -E 'final val'"""], capture_output=True, text=True).stdout)
print(subprocess.run(["bash","-lc","cd /content/memory-native && memory-native-memgate --config tiny "
  "--batch 16 --device cuda --optimizers adamw --seed 0 2>&1 | grep -E 'peak|persistent'"],
  capture_output=True, text=True).stdout)

## What to expect (measured reference, A100)

| kind | step ms | tok/s | persistent | peak VRAM | val loss (2 seeds) |
|---|---:|---:|---:|---:|---:|
| dense + AdamW | 8.1-9.1 | 450-510k (gate 383-399k) | 12.21 MiB | 342.4 MiB | 2.17 |
| counter_rms (old torch path) | 59.6 | 69k (gate 84k) | 3.31 MiB | 324.5 MiB | 1.94 |
| counter_triton (fixed) | 18.4 | 223k (gate ~167k) | 2.56 MiB | 306.8 MiB | 1.96 |

The profiling cell should print `CPU ... | GPU ... | launches/step -> CPU-bound` and show
`_decode_matmul_kernel`/`_decode_gradx_kernel` as the main CUDA ops of the counter kinds.
The verification cell must report all decimation/one-update checks passing and the lr=0
no-op check passing at both shapes. Ratios are the stable quantity; absolute ms depend
on the runtime.